# Chopp & Cia · 01 — Extração

**Projeto Integrador VI** · FATEC Votorantim · 2º Semestre/2026

Converte o backup do ERP num dataset analítico: **uma linha por cliente**.

| | |
|:---|:---|
| **Entrada** | `DB_POWER_SYS*.sql` — dump relacional do ERP |
| **Saída** | `dataset_consolidado_v<DATA_VERSION>.csv` |
| **Ambiente** | Windows local · sem Spark, sem MLflow |
| **Próximo** | 02 publica no Databricks · 03 a 07 leem este CSV |

O dump traz 133 tabelas. Deste notebook saem 15 delas, reduzidas a 43 colunas
por cliente: cadastro, RFM de vendas, atraso de pagamento, atraso de comodato e
os termos CONTRATADOS na venda (condição de pagamento, prazo e valor do
comodato) — explicativos por natureza, porque existem antes de qualquer atraso.

> **Filtros do ERP (v1.1).** O dump é um retrato do banco operacional. Três
> regras precisam ser aplicadas antes de agregar qualquer coisa: `FL_ATIVO = 1`
> (o ERP não apaga registro, marca como inativo), `ID_TIPO_OPERACAO` de venda
> (comodato e retorno moram na mesma tabela dos itens vendidos) e `ID_PESSOA`
> presente em `TB_CLIENTE` (nem toda pessoa do cadastro é cliente). A v1.0 não
> aplicava nenhuma das três — ver a seção 4.

> **Sobre vazamento:** toda coluna nova foi checada uma a uma contra o dump. A
> maior parte do que pareceria útil para crédito — renda mensal, crédito
> disponível, estado civil — está 100% vazia nesta base; não entrou. E
> `FL_BLOQUEADO`/`DS_MOTIVO_BLOQUEIO` ficaram de fora por vazamento confirmado:
> o ERP bloqueia o pedido **porque** o cliente já está inadimplente, e o motivo
> chega a conter o texto literal do atraso.

> **Sem identificação nominal.** `NM_PESSOA` e `DS_FANTASIA` não fazem parte do
> contrato: nome não prevê inadimplência, e o que acrescentaria seria viés — um
> modelo com acesso a nomes aprende clientes específicos em vez de conduta.
> A unidade de análise é `ID_PESSOA` do começo ao fim.

## 1. Painel de controle

Tudo que muda o conteúdo do CSV publicado está nesta célula. As seguintes só leem
daqui — não há número solto no meio do notebook.

Preencha `CAMINHO_SQL` com o caminho do dump. Incremente `DATA_VERSION` sempre
que trocar o dump ou mexer numa regra: cada versão vira um arquivo próprio, e é
isso que mantém reproduzível um resultado de ontem.

> **v1.1** — entram os três filtros que o ERP exige e que a v1.0 não aplicava:
> `FL_ATIVO = 1` (registro excluído), `ID_TIPO_OPERACAO` de venda (o dump traz
> comodato e retorno na mesma tabela dos itens vendidos) e `ID_PESSOA` presente
> em `TB_CLIENTE` (nem toda pessoa é cliente). Os números de RFM da v1.0 estavam
> distorcidos — ver a seção 7.

In [ ]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd

# ── Entrada e saída ──────────────────────────────────────────────────────────
CAMINHO_SQL = r""
PASTA_SAIDA = r""

DATA_VERSION = "1.0"       # incremente ao trocar o dump ou qualquer regra abaixo
SOBRESCREVER = False       # protege um CSV já publicado desta versão

# ── Regras de negócio ────────────────────────────────────────────────────────
# O ERP não apaga: marca FL_ATIVO = 0. Sem este filtro o registro excluído entra
# no dataset como se valesse.
FLAG_ATIVO = "1"

# TB_PEDIDO_ITEM.ID_TIPO_OPERACAO aponta para TB_TIPO_OPERACAO, que tem 47 tipos
# — e só estes cinco são venda. Os demais (comodato, retorno, bonificação,
# remessa, compra, ajuste) são movimentação de estoque com VL_FINANCEIRO = 0:
# não alteram o faturamento, mas contam como pedido e diluem o TICKET_MEDIO.
OPERACOES_VENDA = [1, 11, 12, 30, 38]      # VENDA, VENDA NF, VENDA DE ATIVO,
                                           # VENDA INSUMOS TERCEIROS, VENDA****
# Devolução de venda: mesma natureza comercial, sinal oposto. Entra no
# faturamento subtraindo, e não conta como compra nova na frequência.
OPERACOES_DEVOLUCAO = [14, 15, 41]         # DEVOLUCAO DE VENDA (e variantes)

# ID_STATUS = 9 é CANCELADO. Não é redundante com FL_ATIVO: neste dump 488
# cancelados estão com FL_ATIVO = 0, mas outros 126 seguem com FL_ATIVO = 1.
STATUS_PEDIDO_EXCLUIR = [9]

CORE_INCLUIR = r"CHOPP|CHOPEIRA|BARRIL|CILINDRO|VÁLVULA|VALVULA|EXTRATORA|GÁS|GAS|CIL"
CORE_EXCLUIR = r"COPO|DESCARTÁVEL|DESCARTAVEL|GELO|ÁGUA|AGUA|REFRIGERANTE|SUCO"

LIMITE_RISCO_EDA = 0.20    # rótulo de risco da EDA; o alvo do modelo vive no 04

AGING_FAIXAS = [
    (0, "Sem Atraso"), (3, "1-3 Dias"), (7, "4-7 Dias"), (15, "8-15 Dias"),
    (20, "16-20 Dias"), (30, "21-30 Dias"), (float("inf"), "+30 Dias"),
]

# A fronteira com o Paraguai gera várias grafias do mesmo município.
MAPA_CIDADES = [
    (["PONTA POR", "SANGA", "SANGRA"], "PONTA PORÃ"),
    (["PEDRO JUAN"], "PEDRO JUAN CABALLERO"),
    (["AMAMBA", "AMANBA"], "AMAMBAI"),
]
CIDADES_VAZIAS = ["NAN", "", "NONE", "NÃO PREENCHIDO", "NAO PREENCHIDO"]
CIDADE_DEFAULT = "OUTRAS CIDADES"

MAPA_PERFIL = {"F": "Física", "J": "Jurídica", "E": "Estrangeiro"}

CSV_SAIDA = {"sep": ";", "encoding": "utf-8-sig", "index": False}

# ── Resolução dos caminhos ───────────────────────────────────────────────────
if not CAMINHO_SQL.strip():
    raise ValueError("Preencha CAMINHO_SQL com o caminho do dump .sql do ERP.")

SQL_FILE = Path(CAMINHO_SQL.strip())
if not SQL_FILE.exists():
    raise FileNotFoundError(f"Dump não encontrado: {SQL_FILE}")

OUTPUT_DIR = Path(PASTA_SAIDA.strip()) if PASTA_SAIDA.strip() else SQL_FILE.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DESTINO = OUTPUT_DIR / f"dataset_consolidado_v{DATA_VERSION.replace('.', '_')}.csv"

if CSV_DESTINO.exists() and not SOBRESCREVER:
    raise FileExistsError(
        f"{CSV_DESTINO.name} já existe. Incremente DATA_VERSION ou use SOBRESCREVER = True."
    )

print(f"Origem  : {SQL_FILE.name} ({SQL_FILE.stat().st_size / 1024**2:.0f} MB)")
print(f"Destino : {CSV_DESTINO}")

## 2. Contrato de colunas

Declara explicitamente cada campo consumido. A projeção é aplicada **durante o
parse**: as ~630 colunas fora do contrato nunca chegam à memória.

Este dicionário é a interface com os notebooks seguintes — feature nova lá exige
coluna de origem aqui, e `DATA_VERSION` incrementada.

Nem tudo que entra aqui vira feature: `FL_ATIVO`, `ID_TIPO_OPERACAO` e
`ID_STATUS` são colunas de **controle**, usadas para filtrar na seção 4 e
descartadas em seguida.

In [ ]:
# O dump traz 133 tabelas; TB_PEDIDO_ITEM sozinha tem 200 colunas, quase todas
# fiscais. Só o que está aqui é lido — a projeção acontece durante o parse e o
# resto nunca chega à memória.
#
# NM_PESSOA e DS_FANTASIA estão fora por decisão de projeto: nome não prevê
# inadimplência e o que acrescentaria seria viés. A unidade de análise é
# ID_PESSOA do começo ao fim.
#
# Colunas ADITIVAS ao contrato original (avaliadas uma a uma contra o dump
# real antes de entrar — a maioria dos campos "óbvios" de crédito, como
# VL_RENDA_MENSAL e VL_CREDITO_DISPONIVEL, está 100% vazia nesta base e por
# isso NÃO entrou):
#   TB_PEDIDO_PARCELA.ID_CONDICAO_PAGTO + TB_CONDICAO_PAGTO.TP_CONDICAO
#     → à vista ou a prazo é decidido NA VENDA, antes de qualquer atraso
#       existir. Não é consequência do risco; é o termo contratado.
#   TB_COMODATO.QT_DIAS_EMPRESTIMO, VL_TOTAL_COMODATO
#     → prazo e valor ACORDADOS no contrato, não o desfecho dele.
#   TB_CLIENTE.TP_MODALIDADE, FL_COMISSAO
#     → atributo de cadastro, fixado na criação do cliente.
#
# FL_ATIVO entra em TODAS as tabelas transacionais: o ERP não apaga registro,
# marca FL_ATIVO = 0. É coluna de controle, não feature — sai do dataset depois
# de filtrar, na seção 4.
#
# ID_TIPO_OPERACAO entra em TB_PEDIDO_ITEM pelo mesmo motivo: distingue a venda
# da movimentação de comodato, que vive na MESMA tabela. Também é de controle.
#
# Deliberadamente FORA por vazamento confirmado: TB_PEDIDO.FL_BLOQUEADO e
# DS_MOTIVO_BLOQUEIO. O ERP marca o pedido como bloqueado PORQUE o cliente já
# está em atraso — e DS_MOTIVO_BLOQUEIO chega a conter o texto literal
# "TITULO ABERTO VENCIDO". É o alvo escrito por extenso dentro de uma coluna
# que pareceria explicativa.
CONTRATO = {
    "TB_PESSOA": ["ID_PESSOA", "TP_TIPO", "FL_ATIVO"],
    "TB_CLIENTE": ["ID_CLIENTE", "ID_PESSOA", "ID_TIPO_ESTABELECIMENTO",
                   "ID_FORMA_PAGAMENTO_1", "TP_MODALIDADE", "FL_COMISSAO",
                   "FL_ATIVO"],
    "TB_CLIENTE_ENDERECO": ["ID_CLIENTE", "DS_CIDADE"],
    "TB_FORMA_PAGTO": ["ID_FORMA_PAGTO", "DS_FORMA_PAGTO"],
    "TB_TIPO_ESTABELECIMENTO": ["ID_TIPO_ESTABELECIMENTO", "DS_TIPO_ESTABELECIMENTO"],
    "TB_PRODUTO": ["ID_PRODUTO", "DS_PRODUTO"],
    "TB_CONDICAO_PAGTO": ["ID_CONDICAO_PAGTO", "TP_CONDICAO"],

    "TB_PEDIDO_ITEM": ["ID_PEDIDO", "ID_PRODUTO", "QTD_VENDA", "VL_FINANCEIRO",
                       "ID_TIPO_OPERACAO", "FL_ATIVO"],
    "TB_PEDIDO": ["ID_PEDIDO", "ID_PESSOA", "DT_PEDIDO", "DT_ACERTO", "ID_STATUS",
                  "FL_ATIVO"],

    # Catálogo dos 47 tipos de operação. Entra só para nomear o que foi filtrado
    # no relatório da seção 4 — nenhuma coluna dele chega ao dataset.
    "TB_TIPO_OPERACAO": ["ID_TIPO_OPERACAO", "DS_TIPO_OPERACAO"],

    # Não existe TB_FINANCEIRO neste dump: a parcela vive em
    # TB_CONTAS_A_RECEBER_PARCELA e o vínculo com pessoa/pedido no título.
    "TB_CONTAS_A_RECEBER": ["ID_CONTAS_A_RECEBER", "ID_PESSOA", "ID_PEDIDO", "FL_ATIVO"],
    "TB_CONTAS_A_RECEBER_PARCELA": ["ID_CONTAS_A_RECEBER", "NR_PARCELA", "DT_VENCIMENTO",
                                    "DT_RECEBIMENTO", "DT_BAIXA", "VL_PARCELA",
                                    "VL_RECEBIDO", "TP_BAIXA", "FL_ATIVO"],

    # TB_PEDIDO_PARCELA é distinta de TB_CONTAS_A_RECEBER_PARCELA: a primeira
    # registra o TERMO contratado na venda (condição de pagamento), a segunda
    # o DESFECHO financeiro (pago, atrasado). São trilhas diferentes por
    # desenho — misturar as duas apagaria essa distinção causal.
    "TB_PEDIDO_PARCELA": ["ID_PEDIDO", "ID_CONDICAO_PAGTO"],

    "TB_COMODATO": ["ID_COMODATO", "ID_CLIENTE", "ID_PEDIDO", "DT_EMPRESTIMO",
                    "DT_VENCIMENTO", "DT_RECOLHE", "ID_STATUS",
                    "QT_DIAS_EMPRESTIMO", "VL_TOTAL_COMODATO", "FL_ATIVO"],
    "TB_COMODATO_BEM": ["ID_COMODATO", "ID_PRODUTO", "QTD_PRODUTO", "FL_ATIVO"],
}

## 3. Parser do dump

Lê o `.sql` linha a linha e materializa só as colunas do contrato.

Três armadilhas do formato do SQL Server são tratadas aqui, e todas corrompem
dado **sem lançar erro** se passarem: o `CAST` de tipo parametrizado
`Decimal(18, 2)`, o prefixo `N` de literal Unicode, e o INSERT que ocupa várias
linhas porque um campo de texto contém quebra de linha.

A célula seguinte audita o resultado e **falha a carga** se algum resíduo sobrou.

In [ ]:
RE_INSERT = re.compile(
    r"INSERT\s+\[dbo\]\.\[([A-Z0-9_]+)\]\s*\((.*?)\)\s*VALUES\s*\((.*)\);?", re.IGNORECASE
)
# O tipo do CAST pode vir parametrizado — Decimal(18, 2). Sem o grupo opcional de
# parênteses o regex para no ')' interno e deixa um órfão que desalinha todos os
# campos seguintes.
RE_CAST_TEXTO = re.compile(
    r"CAST\(\s*N?'([^']*)'\s+AS\s+[A-Za-z]+(?:\([^)]*\))?\s*\)", re.IGNORECASE
)
RE_CAST_NUM = re.compile(
    r"CAST\(\s*([^'(),]*?)\s+AS\s+[A-Za-z]+(?:\([^)]*\))?\s*\)", re.IGNORECASE
)
# O csv.reader só honra o quotechar quando a aspa é o PRIMEIRO caractere do campo.
# Em N'CASAS NOTURNAS, BOATES' o primeiro caractere é o N, a aspa não protege nada,
# e a vírgula interna parte o campo em dois — deslocando todas as colunas
# seguintes. Por isso o N sai aqui, ANTES do split, e não só em limpar().
RE_PREFIXO_N = re.compile(r"(?<![A-Za-z0-9_])N'")


def limpar(valor):
    v = valor.strip()
    if v.upper() == "NULL":
        return None
    # Sem remover o N de N'F', TP_TIPO não casa com MAPA_PERFIL e PERFIL vira
    # constante — uma feature morta que nenhum encoder reclama.
    if v[:2].upper() == "N'":
        v = v[2:]
    elif v.startswith("'"):
        v = v[1:]
    if v.endswith("'") and not v.endswith("''"):
        v = v[:-1]
    return v.replace("''", "'").strip()


def registros_insert(caminho):
    """Um INSERT completo por iteração.

    Campos de texto do ERP contêm quebras de linha literais, então um INSERT pode
    ocupar várias linhas do arquivo. O registro só fecha quando as aspas estão
    pareadas e a linha termina em ')'.
    """
    buffer = ""
    with open(caminho, "r", encoding="utf-8", errors="ignore") as f:
        for linha in f:
            if buffer:
                buffer += linha
            elif linha.startswith("INSERT"):
                buffer = linha
            else:
                continue
            if buffer.count("'") % 2 == 0 and buffer.rstrip().rstrip(";").endswith(")"):
                yield buffer
                buffer = ""
    if buffer:
        yield buffer


linhas = {t: [] for t in CONTRATO}
posicoes = {}          # tabela -> [(indice_no_dump, nome_da_coluna)]
n_colunas_dump = {}
descartadas = {t: 0 for t in CONTRATO}

for registro in registros_insert(SQL_FILE):
    registro = registro.replace("\r", " ").replace("\n", " ")
    m = RE_INSERT.search(registro)
    if not m:
        continue

    tabela = m.group(1).upper()
    if tabela not in CONTRATO:
        continue

    if tabela not in posicoes:
        cols = [c.strip().strip("[]") for c in m.group(2).split(",")]
        n_colunas_dump[tabela] = len(cols)
        posicoes[tabela] = [(cols.index(c), c) for c in CONTRATO[tabela] if c in cols]
        ausentes = [c for c in CONTRATO[tabela] if c not in cols]
        if ausentes:
            print(f"  {tabela}: colunas do contrato ausentes no dump -> {ausentes}")

    valores_raw = RE_CAST_NUM.sub(r"\1", RE_CAST_TEXTO.sub(r"'\1'", m.group(3)))
    valores_raw = RE_PREFIXO_N.sub("'", valores_raw)
    for row in csv.reader([valores_raw], delimiter=",", quotechar="'", skipinitialspace=True):
        n = n_colunas_dump[tabela]
        # Campo A MAIS significa que o split partiu um valor no meio: as colunas
        # seguintes estão deslocadas e o registro inteiro é lixo. Truncar em
        # silêncio grava o dado errado como se fosse bom.
        if len(row) > n:
            descartadas[tabela] += 1
            continue
        valores = [limpar(v) for v in row] + [None] * (n - len(row))
        linhas[tabela].append([valores[i] for i, _ in posicoes[tabela]])

db = {
    t: pd.DataFrame(linhas[t], columns=[nome for _, nome in posicoes[t]])
    for t in CONTRATO if linhas[t]
}
del linhas

for t in CONTRATO:
    n = len(db[t]) if t in db else 0
    aviso = (f"   [{descartadas[t]} registro(s) desalinhado(s) descartado(s)]"
             if descartadas[t] else "")
    print(f"{t:<32} {n:>9,}{aviso}")

In [ ]:
# Estas falhas não lançam exceção sozinhas e corrompem o dado adiante.
faltando = [t for t in CONTRATO if t not in db]
if faltando:
    raise RuntimeError(f"Tabelas do contrato ausentes no dump: {faltando}")

# Coluna ausente vira KeyError cru dez células adiante, onde a causa já não é
# visível. O aviso impresso durante o parse não basta: ele não interrompe nada.
colunas_faltando = {
    t: [c for c in CONTRATO[t] if c not in db[t].columns] for t in CONTRATO
}
colunas_faltando = {t: c for t, c in colunas_faltando.items() if c}
if colunas_faltando:
    raise RuntimeError(f"Colunas do contrato ausentes no dump: {colunas_faltando}")


def ocorrencias(padrao, regex):
    achados = {}
    for tabela, df in db.items():
        n = sum(
            int(df[c].astype(str).str.contains(padrao, regex=regex, na=False).sum())
            for c in df.columns if df[c].dtype == "object"
        )
        if n:
            achados[tabela] = n
    return achados


residuo_cast = ocorrencias("CAST(", False)
if residuo_cast:
    raise RuntimeError(
        f"Resíduo de CAST() após o parse: {residuo_cast}. As linhas afetadas estão com "
        "os campos desalinhados — provável tipo parametrizado novo no dump."
    )

residuo_n = ocorrencias(r"^N'", True)
if residuo_n:
    raise RuntimeError(f"Prefixo N' não removido: {residuo_n}. Verifique limpar().")

print("Parse auditado: tabelas completas, sem resíduo de CAST() nem de N'.")

## 4. Filtros do ERP

O dump é um retrato do banco operacional, não uma base analítica. Três regras do
ERP precisam ser aplicadas antes de qualquer agregação — sem elas o dataset conta
como venda coisa que não é venda:

| Regra | Por quê |
|:---|:---|
| `FL_ATIVO = 1` em toda tabela | o ERP não apaga registro, marca como inativo |
| `ID_STATUS != 9` em `TB_PEDIDO` | pedido cancelado não é faturamento |
| `ID_TIPO_OPERACAO` de venda | comodato e retorno moram na tabela dos itens |
| `ID_PESSOA` em `TB_CLIENTE` | nem toda pessoa do cadastro é cliente |

O terceiro é o que mais muda o resultado, e o que menos aparece: a movimentação
de comodato tem `VL_FINANCEIRO = 0`, então **não altera o faturamento** — passa
despercebida por qualquer auditoria de valor. Mas conta como pedido, e é assim
que `TICKET_MEDIO` (gasto ÷ frequência) acaba diluído pela metade.

A devolução de venda é tratada à parte: é comercial, então entra no faturamento
**subtraindo**, mas não conta como compra nova na frequência.

In [ ]:
def num(serie):
    return pd.to_numeric(serie, errors="coerce")


# O ERP não apaga: marca FL_ATIVO = 0. Sem isso o registro excluído entra como se
# valesse — e nada no dataset final denunciaria a origem.
antes_ativo = {t: len(df) for t, df in db.items()}
for t, df in db.items():
    if "FL_ATIVO" in df.columns:
        db[t] = df[df["FL_ATIVO"].astype(str).str.strip() == FLAG_ATIVO].drop(
            columns=["FL_ATIVO"]
        )

print("FL_ATIVO = 1")
for t in sorted(db):
    removidos = antes_ativo[t] - len(db[t])
    if removidos:
        print(f"  {t:<30} {antes_ativo[t]:>8,} -> {len(db[t]):>8,}  (-{removidos:,})")

# ID_STATUS = 9 (CANCELADO) NÃO é redundante com FL_ATIVO: neste dump os 488
# inativos são todos cancelados, mas outros 126 cancelados seguem com
# FL_ATIVO = 1. Filtrar só um dos dois deixa pedido cancelado no faturamento.
n_antes = len(db["TB_PEDIDO"])
db["TB_PEDIDO"] = db["TB_PEDIDO"][
    ~num(db["TB_PEDIDO"]["ID_STATUS"]).isin(STATUS_PEDIDO_EXCLUIR)
]
print(f"\nID_STATUS != 9 (CANCELADO)")
print(f"  {'TB_PEDIDO':<30} {n_antes:>8,} -> {len(db['TB_PEDIDO']):>8,}  "
      f"(-{n_antes - len(db['TB_PEDIDO']):,})")

# Nem toda pessoa do cadastro é cliente: o dump traz fornecedor, transportadora e
# funcionário na mesma TB_PESSOA. Sem este filtro eles viram linha no dataset,
# porque a união da seção 7 é outer.
#
# A base é a INTERSEÇÃO: pessoa ativa E cliente ativo. Os dois cadastros discordam
# entre si — neste dump 3 clientes ativos apontam para uma pessoa com FL_ATIVO = 0.
# Como inativo significa excluído, eles ficam de fora; tomar só um dos lados os
# traria de volta com o cadastro pela metade.
pessoas_ativas = set(num(db["TB_PESSOA"]["ID_PESSOA"]).dropna())
clientes_ativos = set(num(db["TB_CLIENTE"]["ID_PESSOA"]).dropna())
clientes_validos = pessoas_ativas & clientes_ativos

pessoas_totais, clientes_totais = len(db["TB_PESSOA"]), len(db["TB_CLIENTE"])
db["TB_PESSOA"] = db["TB_PESSOA"][num(db["TB_PESSOA"]["ID_PESSOA"]).isin(clientes_validos)]
db["TB_CLIENTE"] = db["TB_CLIENTE"][num(db["TB_CLIENTE"]["ID_PESSOA"]).isin(clientes_validos)]

print(f"\nID_PESSOA em TB_PESSOA ativa E TB_CLIENTE ativa")
print(f"  {'TB_PESSOA':<30} {pessoas_totais:>8,} -> {len(db['TB_PESSOA']):>8,}  "
      f"(-{pessoas_totais - len(db['TB_PESSOA']):,} não-clientes)")
print(f"  {'TB_CLIENTE':<30} {clientes_totais:>8,} -> {len(db['TB_CLIENTE']):>8,}  "
      f"(-{clientes_totais - len(db['TB_CLIENTE']):,} sem pessoa ativa)")

if not clientes_validos:
    raise RuntimeError(
        "Nenhum cliente ativo após os filtros — provável FL_ATIVO com outro "
        "domínio de valores neste dump."
    )


In [ ]:
# O filtro que mais muda o resultado. TB_PEDIDO_ITEM mistura venda e movimentação
# de comodato: são 47 tipos de operação e só cinco são venda. Como a movimentação
# tem VL_FINANCEIRO = 0, ela não mexe no faturamento — some de qualquer auditoria
# de valor — mas conta como pedido e dilui o TICKET_MEDIO pela metade.
itens = db["TB_PEDIDO_ITEM"].copy()
itens["ID_TIPO_OPERACAO"] = num(itens["ID_TIPO_OPERACAO"])

nomes_operacao = (
    db["TB_TIPO_OPERACAO"].assign(ID_TIPO_OPERACAO=lambda d: num(d["ID_TIPO_OPERACAO"]))
    .set_index("ID_TIPO_OPERACAO")["DS_TIPO_OPERACAO"].to_dict()
)

e_venda = itens["ID_TIPO_OPERACAO"].isin(OPERACOES_VENDA)
e_devolucao = itens["ID_TIPO_OPERACAO"].isin(OPERACOES_DEVOLUCAO)

print("ID_TIPO_OPERACAO — o que entra e o que sai")
resumo = (
    itens.assign(VL=num(itens["VL_FINANCEIRO"]))
    .groupby("ID_TIPO_OPERACAO")
    .agg(itens=("ID_PEDIDO", "size"), valor=("VL", "sum"))
    .sort_values("itens", ascending=False)
)
for op, linha in resumo.iterrows():
    if op in OPERACOES_VENDA:
        marca = "VENDA   "
    elif op in OPERACOES_DEVOLUCAO:
        marca = "DEVOLUÇÃO"
    else:
        marca = "  fora  "
    print(f"  [{marca}] {int(op):>3} {nomes_operacao.get(op, '?'):<34} "
          f"{int(linha.itens):>7,} itens  R$ {linha.valor:>13,.2f}")

# A devolução mantém o sinal invertido no valor e some da contagem de frequência:
# devolver não é comprar de novo. QTD_VENDA acompanha o mesmo sinal.
itens["FL_DEVOLUCAO"] = e_devolucao.astype(int)
for coluna in ["VL_FINANCEIRO", "QTD_VENDA"]:
    itens[coluna] = num(itens[coluna]).where(~e_devolucao, -num(itens[coluna]))

n_antes = len(itens)
db["TB_PEDIDO_ITEM"] = itens[e_venda | e_devolucao].drop(columns=["ID_TIPO_OPERACAO"])
print(f"\n  TB_PEDIDO_ITEM  {n_antes:>8,} -> {len(db['TB_PEDIDO_ITEM']):>8,}  "
      f"(-{n_antes - len(db['TB_PEDIDO_ITEM']):,} de movimentação, não de venda)")

del db["TB_TIPO_OPERACAO"]   # catálogo: serviu ao relatório, não vai ao dataset


## 5. Trilhas transacionais

Três trilhas independentes, cada uma no seu grão: **vendas** (item de pedido),
**financeiro** (parcela) e **comodato** (bem cedido). Todas já filtradas pela
seção 4 e restritas a quem é cliente de fato.

A chave do comodato recebe atenção especial: `TB_COMODATO.ID_CLIENTE` tem nome de
uma chave e valores de outra. Conferido contra o dono do pedido, tratá-lo como
`ID_PESSOA` acerta 100% dos contratos, contra 0,9% traduzindo via `TB_CLIENTE`.

In [ ]:
# O item de pedido não tem ID_PESSOA nem DT_PEDIDO: sem o cabeçalho não há RFM.
vendas = db["TB_PEDIDO_ITEM"].merge(
    db["TB_PEDIDO"][["ID_PEDIDO", "ID_PESSOA", "DT_PEDIDO", "DT_ACERTO", "ID_STATUS"]],
    on="ID_PEDIDO", how="inner",
)

financeiro = db["TB_CONTAS_A_RECEBER_PARCELA"].merge(
    db["TB_CONTAS_A_RECEBER"][["ID_CONTAS_A_RECEBER", "ID_PESSOA", "ID_PEDIDO"]],
    on="ID_CONTAS_A_RECEBER", how="inner",
)

# left: contrato sem bem registrado ainda é um comodato em aberto.
comodato = db["TB_COMODATO"].merge(
    db["TB_COMODATO_BEM"][["ID_COMODATO", "ID_PRODUTO", "QTD_PRODUTO"]],
    on="ID_COMODATO", how="left",
)

# TB_COMODATO.ID_CLIENTE tem nome de uma chave e valores de outra: conferido
# contra o dono do pedido, tratá-lo como ID_PESSOA acerta 100% dos contratos,
# contra 0,9% traduzindo via TB_CLIENTE. Traduzir atribuiria a inadimplência ao
# cliente errado, porque os dois espaços de ID se sobrepõem.
comodato = comodato.rename(columns={"ID_CLIENTE": "ID_PESSOA"})

# As três trilhas ficam restritas a quem é cliente de fato. O título a receber e
# o contrato de comodato de um não-cliente existem no ERP, mas não pertencem a
# nenhuma linha deste dataset — sem o corte aqui, eles entrariam pela união
# outer da seção 7 como cliente fantasma, com cadastro vazio.
n_vendas, n_fin, n_com = len(vendas), len(financeiro), len(comodato)
vendas = vendas[num(vendas["ID_PESSOA"]).isin(clientes_validos)]
financeiro = financeiro[num(financeiro["ID_PESSOA"]).isin(clientes_validos)]
comodato = comodato[num(comodato["ID_PESSOA"]).isin(clientes_validos)]
print(f"restrição a clientes: vendas -{n_vendas - len(vendas):,} · "
      f"financeiro -{n_fin - len(financeiro):,} · comodato -{n_com - len(comodato):,}")

dono_pedido = vendas[["ID_PEDIDO", "ID_PESSOA"]].drop_duplicates("ID_PEDIDO")
conferencia = comodato.merge(
    dono_pedido.rename(columns={"ID_PESSOA": "DONO_PEDIDO"}), on="ID_PEDIDO", how="inner"
)
COERENCIA_COMODATO = (
    float((num(conferencia["ID_PESSOA"]) == num(conferencia["DONO_PEDIDO"])).mean() * 100)
    if len(conferencia) else float("nan")
)
if COERENCIA_COMODATO < 95:
    print(f"  ATENÇÃO: coerência da chave do comodato em {COERENCIA_COMODATO:.1f}% — "
          "reveja a semântica de TB_COMODATO.ID_CLIENTE neste dump.")

print(f"vendas     {len(vendas):>8,} linhas (grão: item)")
print(f"financeiro {len(financeiro):>8,} linhas (grão: parcela)")
print(f"comodato   {len(comodato):>8,} linhas (grão: bem)")
print(f"chave do comodato coerente com o dono do pedido: {COERENCIA_COMODATO:.1f}%")

# Quarta trilha: o TERMO de pagamento contratado na venda (à vista/a prazo),
# não o desfecho financeiro. TP_CONDICAO já existe na hora do pedido — é
# explicativo por natureza, ao contrário de TAXA_ATRASO_PAGAMENTO.
condicao = db["TB_PEDIDO_PARCELA"].merge(
    db["TB_PEDIDO"][["ID_PEDIDO", "ID_PESSOA"]], on="ID_PEDIDO", how="inner",
).merge(
    db["TB_CONDICAO_PAGTO"], on="ID_CONDICAO_PAGTO", how="left",
)
condicao = condicao[num(condicao["ID_PESSOA"]).isin(clientes_validos)]
print(f"condição   {len(condicao):>8,} linhas (grão: parcela contratada)")

## 6. Agregação por cliente

Cada trilha é agregada **na sua própria granularidade** antes da união — e essa
ordem é o ponto central da consolidação.

Encadear as três por `ID_PEDIDO` multiplicaria as linhas (N itens × M parcelas ×
K bens: 24.399 → 43.562 neste dump) e contaria o mesmo faturamento várias vezes,
sem que nada indicasse o erro. A auditoria confere que o valor se conserva ao
centavo.

In [ ]:
# Cada trilha é agregada na sua própria granularidade ANTES da união. Encadear os
# três merges por ID_PEDIDO multiplicaria as linhas (N itens × M parcelas × K
# bens) e contaria o mesmo VL_FINANCEIRO várias vezes.
v = vendas.copy()
v["DT_PEDIDO"] = pd.to_datetime(v["DT_PEDIDO"], errors="coerce")
v["VL_FINANCEIRO"] = num(v["VL_FINANCEIRO"])
v["QTD_VENDA"] = num(v["QTD_VENDA"])

DATA_CORTE = v["DT_PEDIDO"].max()
print(f"Data de corte (último pedido): {DATA_CORTE:%d/%m/%Y}")

# Duas visões do mesmo grão. O VALOR é líquido — a devolução entra negativa,
# porque faturamento devolvido não é faturamento. Já a CONTAGEM (frequência,
# itens, datas) olha só a compra: devolver não é comprar de novo, e a data da
# devolução não é a data da última compra.
compras = v[v["FL_DEVOLUCAO"] == 0]

agg_vendas = compras.groupby("ID_PESSOA").agg(
    PRIMEIRA_COMPRA=("DT_PEDIDO", "min"),
    ULTIMA_COMPRA=("DT_PEDIDO", "max"),
    FREQUENCIA_COMPRAS=("ID_PEDIDO", "nunique"),
    TOTAL_ITENS=("ID_PEDIDO", "count"),
).reset_index()

# O valor vem de v (com devolução), a contagem de compras: são universos
# diferentes, então quem só devolveu aparece no valor sem entrar na frequência.
agg_valor = v.groupby("ID_PESSOA").agg(
    TOTAL_GASTO=("VL_FINANCEIRO", "sum"),
    QTD_TOTAL_VENDIDA=("QTD_VENDA", "sum"),
).reset_index()
agg_vendas = agg_vendas.merge(agg_valor, on="ID_PESSOA", how="outer")
agg_vendas[["FREQUENCIA_COMPRAS", "TOTAL_ITENS"]] = (
    agg_vendas[["FREQUENCIA_COMPRAS", "TOTAL_ITENS"]].fillna(0)
)

# replace(0, np.nan): sem isso, quem só tem devolução divide por zero e vira inf.
agg_vendas["TICKET_MEDIO"] = (
    agg_vendas["TOTAL_GASTO"] / agg_vendas["FREQUENCIA_COMPRAS"].replace(0, np.nan)
).fillna(0)
agg_vendas["DIAS_DESDE_PRIMEIRA_COMPRA"] = (DATA_CORTE - agg_vendas["PRIMEIRA_COMPRA"]).dt.days
agg_vendas["DIAS_DESDE_ULTIMA_COMPRA"] = (DATA_CORTE - agg_vendas["ULTIMA_COMPRA"]).dt.days

favorito = (
    compras.groupby(["ID_PESSOA", "ID_PRODUTO"])["QTD_VENDA"].sum().reset_index()
    .sort_values(["ID_PESSOA", "QTD_VENDA"], ascending=[True, False])
    .drop_duplicates("ID_PESSOA")
    .merge(db["TB_PRODUTO"][["ID_PRODUTO", "DS_PRODUTO"]], on="ID_PRODUTO", how="left")
    [["ID_PESSOA", "DS_PRODUTO"]].rename(columns={"DS_PRODUTO": "PRODUTO_FAVORITO"})
)
agg_vendas = agg_vendas.merge(favorito, on="ID_PESSOA", how="left")

# Core business depende de ID_PRODUTO, que não sobrevive à agregação: a marcação
# precisa sair daqui, do grão transacional.
produtos = db["TB_PRODUTO"].copy()
produtos["DS_PRODUTO"] = produtos["DS_PRODUTO"].fillna("").str.upper()
ids_core = produtos[
    produtos["DS_PRODUTO"].str.contains(CORE_INCLUIR, regex=True)
    & ~produtos["DS_PRODUTO"].str.contains(CORE_EXCLUIR, regex=True)
]["ID_PRODUTO"].unique()
clientes_core = set(compras[compras["ID_PRODUTO"].isin(ids_core)]["ID_PESSOA"].dropna())

print(f"core business: {len(ids_core)} produtos, {len(clientes_core):,} clientes")

In [ ]:
# Atrasou = pagou depois do vencimento OU não pagou e o vencimento já passou.
# Sem a segunda metade, quem nunca pagou contaria como adimplente por não ter
# data de baixa.
f = financeiro.copy()
f["DT_VENCIMENTO"] = pd.to_datetime(f["DT_VENCIMENTO"], errors="coerce")
col_pagamento = "DT_BAIXA" if "DT_BAIXA" in f.columns else "DT_RECEBIMENTO"
f[col_pagamento] = pd.to_datetime(f[col_pagamento], errors="coerce")
f["VL_PARCELA"] = num(f["VL_PARCELA"])

f["DIAS_ATRASO"] = (
    f[col_pagamento].fillna(DATA_CORTE) - f["DT_VENCIMENTO"]
).dt.days.clip(lower=0)
f["ATRASOU"] = (
    (f[col_pagamento] > f["DT_VENCIMENTO"])
    | (f[col_pagamento].isna() & (f["DT_VENCIMENTO"] < DATA_CORTE))
).astype(int)

agg_fin = f.groupby("ID_PESSOA").agg(
    # count, não nunique: o grão aqui é a parcela, e um título tem várias.
    TOTAL_PARCELAS=("ID_CONTAS_A_RECEBER", "count"),
    PARCELAS_ATRASADAS=("ATRASOU", "sum"),
    MEDIA_DIAS_ATRASO_PAG=("DIAS_ATRASO", "mean"),
    MAX_DIAS_ATRASO_PAG=("DIAS_ATRASO", "max"),
    VALOR_TOTAL_PARCELAS=("VL_PARCELA", "sum"),
).reset_index()
# replace(0, 1) no denominador: quem tem 0 parcelas fica com taxa 0, não NaN.
agg_fin["TAXA_ATRASO_PAGAMENTO"] = (
    agg_fin["PARCELAS_ATRASADAS"] / agg_fin["TOTAL_PARCELAS"].replace(0, 1)
)

c = comodato.copy()
c["DT_VENCIMENTO"] = pd.to_datetime(c["DT_VENCIMENTO"], errors="coerce")
c["DT_RECOLHE"] = pd.to_datetime(c["DT_RECOLHE"], errors="coerce")
c["QTD_PRODUTO"] = num(c["QTD_PRODUTO"])
c["QT_DIAS_EMPRESTIMO"] = num(c["QT_DIAS_EMPRESTIMO"])
c["VL_TOTAL_COMODATO"] = num(c["VL_TOTAL_COMODATO"])

c["DIAS_ATRASO"] = (
    c["DT_RECOLHE"].fillna(DATA_CORTE) - c["DT_VENCIMENTO"]
).dt.days.clip(lower=0)
c["ATRASOU"] = (
    (c["DT_RECOLHE"] > c["DT_VENCIMENTO"])
    | (c["DT_RECOLHE"].isna() & (c["DT_VENCIMENTO"] < DATA_CORTE))
).astype(int)

# Unidades somam por bem, mas o risco conta por CONTRATO: sem o dedup, um
# comodato com 3 chopeiras contaria 3 atrasos. PRAZO_MEDIO_COMODATO e
# VALOR_MEDIO_COMODATO usam o mesmo dedup: são atributos do CONTRATO
# (acordados na assinatura), não do bem.
qtd_equipamentos = c.groupby("ID_PESSOA")["QTD_PRODUTO"].sum().rename("QTD_EQUIPAMENTOS")
agg_com = c.drop_duplicates("ID_COMODATO").groupby("ID_PESSOA").agg(
    TOTAL_COMODATOS=("ID_COMODATO", "nunique"),
    COMODATOS_ATRASADOS=("ATRASOU", "sum"),
    MEDIA_DIAS_ATRASO_COM=("DIAS_ATRASO", "mean"),
    MAX_DIAS_ATRASO_COM=("DIAS_ATRASO", "max"),
    PRAZO_MEDIO_COMODATO=("QT_DIAS_EMPRESTIMO", "mean"),
    VALOR_MEDIO_COMODATO=("VL_TOTAL_COMODATO", "mean"),
).reset_index().merge(qtd_equipamentos, on="ID_PESSOA", how="left")
agg_com["TAXA_ATRASO_COMODATO"] = (
    agg_com["COMODATOS_ATRASADOS"] / agg_com["TOTAL_COMODATOS"].replace(0, 1)
)

# Termo de pagamento contratado (à vista / a prazo). TP_CONDICAO vem de
# TB_CONDICAO_PAGTO: 'V' = à vista, 'P' = a prazo. Fixado na venda, então não
# carrega nenhuma informação sobre se a parcela foi paga depois.
cnd = condicao.copy()
cnd["A_PRAZO"] = (cnd["TP_CONDICAO"] == "P").astype(int)
agg_condicao = cnd.groupby("ID_PESSOA").agg(
    PCT_COMPRAS_A_PRAZO=("A_PRAZO", "mean"),
).reset_index()

print(f"clientes agregados — vendas {len(agg_vendas):,} · financeiro {len(agg_fin):,} "
      f"· comodato {len(agg_com):,} · condição {len(agg_condicao):,}")

## 7. Dimensões cadastrais e união

Junta o cadastro às três trilhas agregadas e deriva as colunas de apoio à EDA:
faixas de aging, flags de cobertura e `PERFIL_RISCO`.

A união é `outer` para não perder o cliente que existe em uma trilha só. Onde a
ausência significa zero (nenhuma parcela são 0 parcelas), preenche com zero; as
datas ficam nulas de propósito — quem nunca comprou não tem primeira compra.

In [ ]:
def agrupar_cidade(cidade):
    c = str(cidade).upper().strip()
    for chaves, destino in MAPA_CIDADES:
        if any(k in c for k in chaves):
            return destino
    return "NÃO PREENCHIDO" if c in CIDADES_VAZIAS else CIDADE_DEFAULT


def faixa_aging(dias):
    d = pd.to_numeric(dias, errors="coerce")
    if pd.isna(d) or d <= 0:
        return AGING_FAIXAS[0][1]
    for limite, rotulo in AGING_FAIXAS:
        if d <= limite:
            return rotulo
    return AGING_FAIXAS[-1][1]


dim_pessoa = db["TB_PESSOA"][["ID_PESSOA", "TP_TIPO"]].drop_duplicates("ID_PESSOA").copy()
dim_pessoa["PERFIL"] = dim_pessoa["TP_TIPO"].map(MAPA_PERFIL).fillna("Não Informado")
dim_pessoa = dim_pessoa.drop(columns=["TP_TIPO"])

# O dedup vem ANTES dos merges: um ID_PESSOA com dois ID_CLIENTE fica com o
# cadastro do primeiro registro do dump. Escolha arbitrária, mas assumida.
cliente = db["TB_CLIENTE"][
    ["ID_CLIENTE", "ID_PESSOA", "ID_TIPO_ESTABELECIMENTO", "ID_FORMA_PAGAMENTO_1",
     "TP_MODALIDADE", "FL_COMISSAO"]
].drop_duplicates("ID_PESSOA").merge(
    db["TB_FORMA_PAGTO"][["ID_FORMA_PAGTO", "DS_FORMA_PAGTO"]],
    left_on="ID_FORMA_PAGAMENTO_1", right_on="ID_FORMA_PAGTO", how="left",
).merge(
    db["TB_TIPO_ESTABELECIMENTO"][["ID_TIPO_ESTABELECIMENTO", "DS_TIPO_ESTABELECIMENTO"]],
    on="ID_TIPO_ESTABELECIMENTO", how="left",
)
cliente["PAGAMENTO"] = cliente["DS_FORMA_PAGTO"].fillna("OUTROS").str.upper()
cliente["SEGMENTO"] = cliente["DS_TIPO_ESTABELECIMENTO"].fillna("OUTROS")
cliente["MODALIDADE"] = cliente["TP_MODALIDADE"].fillna("NÃO INFORMADO")
cliente["TEM_COMISSAO"] = num(cliente["FL_COMISSAO"]).fillna(0)

endereco = db["TB_CLIENTE_ENDERECO"][["ID_CLIENTE", "DS_CIDADE"]].drop_duplicates(
    "ID_CLIENTE"
).copy()
endereco["CIDADE"] = endereco["DS_CIDADE"].apply(agrupar_cidade)

dim_cliente = (
    cliente[["ID_CLIENTE", "ID_PESSOA", "PAGAMENTO", "SEGMENTO", "MODALIDADE", "TEM_COMISSAO"]]
    .merge(endereco[["ID_CLIENTE", "CIDADE"]], on="ID_CLIENTE", how="left")
    .drop_duplicates("ID_PESSOA")
    .drop(columns=["ID_CLIENTE"])
)

# outer: não perder o cliente que existe em uma trilha só.
dataset = (
    dim_pessoa.merge(dim_cliente, on="ID_PESSOA", how="outer")
    .merge(agg_vendas, on="ID_PESSOA", how="outer")
    .merge(agg_fin, on="ID_PESSOA", how="outer")
    .merge(agg_com, on="ID_PESSOA", how="outer")
    .merge(agg_condicao, on="ID_PESSOA", how="outer")
)

# Zero é a leitura correta: "nenhuma parcela" são 0 parcelas. As datas ficam
# nulas de propósito — quem nunca comprou não tem primeira compra.
# PRAZO_MEDIO_COMODATO e VALOR_MEDIO_COMODATO ficam de fora do zero-fill de
# propósito: quem não tem comodato não tem prazo nem valor médio — são nulos
# genuínos, não zero, para não sugerir "comodato de prazo 0".
COLS_ZERO = [
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "TOTAL_GASTO", "QTD_TOTAL_VENDIDA", "TICKET_MEDIO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG",
    "VALOR_TOTAL_PARCELAS", "TAXA_ATRASO_PAGAMENTO",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM",
    "QTD_EQUIPAMENTOS", "TAXA_ATRASO_COMODATO", "TEM_COMISSAO", "PCT_COMPRAS_A_PRAZO",
]
dataset[COLS_ZERO] = dataset[COLS_ZERO].fillna(0)
for coluna, valor in [("PERFIL", "Não Informado"), ("CIDADE", "NÃO PREENCHIDO"),
                      ("PAGAMENTO", "OUTROS"), ("SEGMENTO", "OUTROS"),
                      ("MODALIDADE", "NÃO INFORMADO")]:
    dataset[coluna] = dataset[coluna].fillna(valor)

dataset["CORE_BUSINESS"] = dataset["ID_PESSOA"].isin(clientes_core).astype(int)
dataset["TEM_VENDAS"] = (dataset["FREQUENCIA_COMPRAS"] > 0).astype(int)
dataset["TEM_FINANCEIRO"] = (dataset["TOTAL_PARCELAS"] > 0).astype(int)
dataset["TEM_COMODATO"] = (dataset["TOTAL_COMODATOS"] > 0).astype(int)

# Aging pela severidade MÁXIMA: um atraso de 45 dias é caso de "+30 Dias" ainda
# que a média o diluísse numa faixa branda.
dataset["AGING_PAGAMENTO"] = dataset["MAX_DIAS_ATRASO_PAG"].apply(faixa_aging)
dataset["AGING_COMODATO"] = dataset["MAX_DIAS_ATRASO_COM"].apply(faixa_aging)
dataset["MES_ULTIMA_COMPRA"] = (
    pd.to_datetime(dataset["ULTIMA_COMPRA"], errors="coerce").dt.to_period("M").astype(str)
)

dataset["RISCO_FINANCEIRO"] = (dataset["TAXA_ATRASO_PAGAMENTO"] > LIMITE_RISCO_EDA).astype(int)
dataset["RISCO_COMODATO"] = (dataset["TAXA_ATRASO_COMODATO"] > LIMITE_RISCO_EDA).astype(int)
dataset["PERFIL_RISCO"] = np.select(
    [
        (dataset["RISCO_FINANCEIRO"] == 1) & (dataset["RISCO_COMODATO"] == 1),
        dataset["RISCO_FINANCEIRO"] == 1,
        dataset["RISCO_COMODATO"] == 1,
    ],
    ["RISCO DUPLO", "SÓ FINANCEIRO", "SÓ COMODATO"],
    default="SEM RISCO",
)

dataset["_DATA_VERSION"] = DATA_VERSION

COLUNAS = [
    "ID_PESSOA", "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO", "MODALIDADE", "TEM_COMISSAO",
    "PRIMEIRA_COMPRA", "ULTIMA_COMPRA", "MES_ULTIMA_COMPRA",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
    "PRODUTO_FAVORITO", "PCT_COMPRAS_A_PRAZO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS", "AGING_PAGAMENTO",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS", "AGING_COMODATO",
    "PRAZO_MEDIO_COMODATO", "VALOR_MEDIO_COMODATO",
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
    "_DATA_VERSION",
]
dataset = dataset[COLUNAS]

print(f"dataset consolidado: {len(dataset):,} clientes × {dataset.shape[1]} colunas")

## 8. Auditoria

Seis verificações que **interrompem a carga** em vez de deixar o erro seguir
para os próximos notebooks, onde a causa já não seria visível:

| Verificação | O que detecta |
|:---|:---|
| conservação de valor | duplicação de linhas em algum merge |
| chave única | quebra da promessa de 1 linha por cliente |
| categóricas vivas | feature degradada a constante pelo prefixo `N'` |
| sem identificação nominal | coluna de nome reintroduzida por carga futura |
| frequência ≤ pedidos de venda | movimentação de comodato contada como compra |
| todo cliente é cliente | pessoa não-cliente vazando pela união outer |

As duas últimas são novas na v1.1 e existem por causa do defeito que ela corrige.
A conservação de valor **não** teria pego aquele erro: a movimentação de comodato
tem `VL_FINANCEIRO = 0`, então o faturamento batia ao centavo enquanto a
frequência contava 16.629 itens que não eram venda.

In [ ]:
# Estas falham a carga em vez de deixar o erro seguir para os próximos notebooks,
# onde a causa já não seria visível.
faturamento_origem = float(v["VL_FINANCEIRO"].sum())
faturamento_consolidado = float(dataset["TOTAL_GASTO"].sum())
if abs(faturamento_origem - faturamento_consolidado) > 0.01:
    raise RuntimeError(
        f"Faturamento não conservado: origem R$ {faturamento_origem:,.2f} vs consolidado "
        f"R$ {faturamento_consolidado:,.2f}. Provável duplicação de linhas em algum merge."
    )

# Esta é a verificação que teria pego o defeito da v1.0. O faturamento conservava
# porque a movimentação de comodato vale R$ 0 — só a CONTAGEM denunciava.
pedidos_de_venda = int(compras["ID_PEDIDO"].nunique())
frequencia_total = int(dataset["FREQUENCIA_COMPRAS"].sum())
if frequencia_total > pedidos_de_venda:
    raise RuntimeError(
        f"Frequência somada ({frequencia_total:,}) maior que os pedidos de venda "
        f"({pedidos_de_venda:,}): há pedido que não é venda contado como compra. "
        "Verifique OPERACOES_VENDA na seção 1."
    )

fora_do_cadastro = set(dataset["ID_PESSOA"].dropna()) - {
    str(p) for p in db["TB_CLIENTE"]["ID_PESSOA"].dropna()
}
if fora_do_cadastro:
    raise RuntimeError(
        f"{len(fora_do_cadastro):,} ID_PESSOA no dataset não estão em TB_CLIENTE. "
        "Nem toda pessoa é cliente — a união outer da seção 7 deixou passar."
    )

if dataset["ID_PESSOA"].duplicated().any():
    raise RuntimeError(
        f"{int(dataset['ID_PESSOA'].duplicated().sum())} ID_PESSOA duplicado(s): "
        "a tabela deve ter uma linha por cliente."
    )

constantes = [c for c in ["PERFIL", "CIDADE", "PAGAMENTO"] if dataset[c].nunique() <= 1]
if constantes:
    raise RuntimeError(
        f"Categóricas constantes: {constantes}. Sintoma do prefixo N' não removido — "
        "treinam sem erro e não informam nada."
    )

PADROES_NOMINAIS = ("NOME", "NM_", "FANTASIA", "RAZAO", "CPF", "CNPJ", "EMAIL",
                    "TELEFONE", "ENDERECO")
nominais = [c for c in dataset.columns if any(p in c.upper() for p in PADROES_NOMINAIS)]
if nominais:
    raise RuntimeError(f"Identificação nominal no consolidado: {nominais}")

print(f"faturamento conservado   R$ {faturamento_consolidado:,.2f} (líquido de devolução)")
print(f"chave única              {len(dataset):,} clientes")
print(f"frequência coerente      {frequencia_total:,} compras <= {pedidos_de_venda:,} "
      f"pedidos de venda")
print(f"todos são clientes       0 ID_PESSOA fora de TB_CLIENTE")
print(f"categóricas com variação PERFIL {dataset['PERFIL'].nunique()} · "
      f"CIDADE {dataset['CIDADE'].nunique()} · PAGAMENTO {dataset['PAGAMENTO'].nunique()}")
print()
for flag, rotulo in [("TEM_VENDAS", "com vendas"), ("TEM_FINANCEIRO", "com financeiro"),
                     ("TEM_COMODATO", "com comodato"), ("CORE_BUSINESS", "core business")]:
    n = int(dataset[flag].sum())
    print(f"{rotulo:<16} {n:>6,} ({n / len(dataset) * 100:>5.1f}%)")

sem_venda = int((dataset["TEM_VENDAS"] == 0).sum())
if sem_venda:
    print(f"\n{sem_venda:,} cliente(s) sem venda têm PRIMEIRA_COMPRA, ULTIMA_COMPRA e "
          "DIAS_DESDE_* nulos — é a leitura correta, não um defeito.")

## 9. Publicação

Grava o CSV consolidado. É a entrada do notebook 02 (que o publica como tabela no
Databricks) e dos notebooks 03 a 07 na execução local.

In [ ]:
dataset.to_csv(CSV_DESTINO, **CSV_SAIDA)

print(CSV_DESTINO)
print(f"{len(dataset):,} clientes × {dataset.shape[1]} colunas · "
      f"{CSV_DESTINO.stat().st_size / 1024**2:.1f} MB")